# Forecasting the licensed steel series, stainless and epoxy

The free IMF/Eurostat series only give **proxies** for steel, stainless and plastics.
Three workbooks in `data/` hold the real prices (EUR/t), which this notebook
forecasts:

| Series | Workbook | BOM material | Missing catalog ID |
|---|---|---|---|
| MEPS Germany CR coil | `Copia de Europe Steel INDEXES.xlsx` | Carbon steel (CR coil) | L01 |
| CRU Germany CR coil (monthly average) | same | Carbon steel (CR coil) | L01 |
| MEPS Germany HDG coil | same | Galvanised steel (HDG) | L03 |
| CRU Germany HDG coil (monthly average) | same | Galvanised steel (HDG) | L03 |
| CRU Stainless 304 CR-2mm Germany (base + alloy) | same | Stainless steel 304L | L02 |
| Epoxy resin and its cost drivers (benzene, acetone, epichlorohydrin) | `Epoxy for student.xlsx` | Epoxy resin EU (BOM line, plastics family) | none |

**Licensing:** MEPS and CRU are licensed. HANDOFF.md lists licensed-data permissions
as an open item with the data owner. This notebook is saved without outputs, so nothing
licensed is stored in it; do not commit executed copies or exported prices.

**Method** (same as `multivariate_var_vecm.ipynb`): rolling-origin backtests,
expanding window, log prices, errors in log points, relative MAE against LOCF, and a
Diebold-Mariano style test with HAC standard errors. Models: LOCF, 12-month drift,
ARIMA(1,1,0), VAR on log-differences and a rank-1 VECM.

**Publication lags are assumptions.** The repo documents lags for the IMF and Eurostat
series only. For the licensed and epoxy series we **assume a 1-month release lag**
(an offset of 2 months, the same as the repo's provisional ECB FX rule). IMF series
keep the repo's offset of 3. Each system is cut to the last month all its series are
available. These are assumptions, not verified release dates.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import openpyxl
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from statsmodels.tsa.api import VAR
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.vector_ar.vecm import VECM

root = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src/schneider_electric_forecasting").exists()
)
sys.path.insert(0, str(root / "src"))

from schneider_electric_forecasting import _01_ingestion as ingestion  # noqa: E402
from schneider_electric_forecasting import config  # noqa: E402
from schneider_electric_forecasting._04_features import PHASE3_LAGS  # noqa: E402

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
DATA = Path(config.DATA_DIR)

## 1. Load the workbooks

The steel workbook has one sheet per provider with multi-row headers, so columns are
picked by position and checked against their header labels before use. Dates are
mapped to the start of the month. Only 2016 onwards is used, to match the other
series. Future rows are blank templates and are ignored.

In [ ]:
STEEL = DATA / "Copia de Europe Steel INDEXES.xlsx"
EPOXY = DATA / "Epoxy for student.xlsx"


def sheet_rows(path, sheet):
    """Read every row of one worksheet as a list of tuples."""
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    try:
        return list(wb[sheet].iter_rows(values_only=True))
    finally:
        wb.close()


def month_series(rows, col):
    """Collect numeric values of one column, indexed by month start, from 2016."""
    values = {}
    for row in rows:
        if hasattr(row[0], "year") and isinstance(row[col], int | float):
            values[pd.Timestamp(row[0].year, row[0].month, 1)] = float(row[col])
    series = pd.Series(values).sort_index()
    return series[series.index >= "2016-01-01"]


def label(rows, col, upto=5):
    """Join the header cells above one column, for checking."""
    return " | ".join(str(r[col]).replace("\n", " ") for r in rows[:upto] if r[col])


meps = sheet_rows(STEEL, "MEPS")
flat = sheet_rows(STEEL, "CRU Flat")
stainless = sheet_rows(STEEL, "CRU Stainless")

# (name, rows, column, expected header fragments)
SPECS = {
    "MEPS_CR_DE": (meps, 2, ("CR coil",)),
    "MEPS_HDG_DE": (meps, 3, ("HDG coil",)),
    "CRU_CR_DE": (flat, 12, ("Germany", "CR coil", "AVERAGE")),
    "CRU_HDG_DE": (flat, 18, ("Germany", "HDG coil", "AVERAGE")),
}
# MEPS lists Germany as the first country block (columns 1-4) on the sheet title row.
assert str(meps[0][1]).strip() == "Germany"
steel = {}
for name, (rows, col, expect) in SPECS.items():
    head = label(rows, col).lower()
    assert all(e.lower() in head for e in expect), (name, head)
    steel[name] = month_series(rows, col)

# Stainless: base + alloy surcharge for Grade 304, CR 2 mm, Germany (columns 1 and 2).
assert "Grade 304" in label(stainless, 1) and "BASE" in label(stainless, 1)
assert "Alloy" in label(stainless, 2)
base, alloy = month_series(stainless, 1), month_series(stainless, 2)
steel["SS304_DE"] = (base + alloy).dropna()

ep = sheet_rows(EPOXY, "Feuil1")
header = ep[0]
assert "EPOXY" in str(header[6]) and "BENZENE" in str(header[2])
epoxy = {}
for name, col in {"EPOXY": 6, "BENZENE": 2, "ACETONE": 3, "ECH": 4}.items():
    epoxy[name] = month_series(ep, col)

licensed = pd.DataFrame({**steel, **epoxy})
summary = pd.DataFrame(
    {
        "first": licensed.apply(lambda s: s.first_valid_index().strftime("%Y-%m")),
        "last": licensed.apply(lambda s: s.last_valid_index().strftime("%Y-%m")),
        "n": licensed.count(),
        "missing inside range": licensed.apply(
            lambda s: (
                len(
                    pd.date_range(
                        s.first_valid_index(), s.last_valid_index(), freq="MS"
                    )
                )
                - s.count()
            )
        ),
    }
)
summary

Stainless 304 is reported as a **base price plus an alloy surcharge** (the surcharge
follows nickel, so it is the volatile part). We model the sum, the price a buyer
pays. The epoxy series is a **calculated** cost build (raw materials plus a computed
"other costs" term), not an observed market quote, so forecasting it is forecasting a
model output.

## 2. Free drivers

Nickel (`S03`), iron ore (`S04`) and coal (`E08`) come from the repo's loaders. They
have the IMF publication lag (offset 3).

In [ ]:
rm = ingestion.load_rm_monthly_long()
en = ingestion.load_energy_monthly_long()
free = (
    pd.concat(
        [rm[["month", "series_id", "value"]], en[["month", "series_id", "value"]]]
    )
    .pivot(index="month", columns="series_id", values="value")
    .sort_index()[["S03", "S04", "S09", "E08"]]
)
free.index = pd.DatetimeIndex(free.index).to_period("M").to_timestamp()

panel = licensed.join(free, how="outer")
logs = np.log(panel)
NAMES = {
    "MEPS_CR_DE": "CR coil, MEPS Germany",
    "MEPS_HDG_DE": "HDG coil, MEPS Germany",
    "CRU_CR_DE": "CR coil, CRU Germany",
    "CRU_HDG_DE": "HDG coil, CRU Germany",
    "SS304_DE": "Stainless 304 (base+alloy), CRU Germany",
    "EPOXY": "Epoxy resin, Europe",
    "BENZENE": "Benzene, Europe",
    "ACETONE": "Acetone, Europe",
    "ECH": "Epichlorohydrin, Europe",
    "S03": "Nickel, LME",
    "S04": "Iron ore",
    "S09": "EU steel PPI (free proxy)",
    "E08": "Coal",
}

LICENSED_OFFSET = 2  # assumed 1-month release lag (+1 incomplete month)
OFFSET = {s: LICENSED_OFFSET for s in licensed.columns}
OFFSET.update({s: PHASE3_LAGS[s].maximum_lag_months + 1 for s in free.columns})

SYSTEMS = {
    "steel_flat": ["MEPS_CR_DE", "MEPS_HDG_DE", "CRU_CR_DE", "CRU_HDG_DE"],
    "steel_drivers": ["MEPS_CR_DE", "S04", "E08"],
    "steel_proxy_link": ["MEPS_CR_DE", "S09"],
    "stainless_nickel": ["SS304_DE", "S03"],
    "epoxy_chain": ["EPOXY", "BENZENE", "ACETONE", "ECH"],
}
SYSTEM_OFFSET = {k: max(OFFSET[s] for s in v) for k, v in SYSTEMS.items()}
print("System offsets (months):", SYSTEM_OFFSET)
for k, v in SYSTEMS.items():
    d = logs[v].dropna()
    print(
        f"{k}: {len(d)} common months, {d.index.min():%Y-%m} to {d.index.max():%Y-%m}"
    )

## 3. Look at the series

Each series is plotted in an index (first month = 100) so shapes are comparable
without showing price levels.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, group in zip(
    axes,
    (
        ["MEPS_CR_DE", "MEPS_HDG_DE", "CRU_CR_DE", "CRU_HDG_DE", "SS304_DE"],
        ["EPOXY", "BENZENE", "ACETONE", "ECH"],
    ),
    strict=True,
):
    for s in group:
        x = panel[s].dropna()
        ax.plot(x.index, 100 * x / x.iloc[0], label=s)
    ax.legend(fontsize=8)
    ax.set_ylabel("index, first month = 100")
plt.show()

## 4. Models

All models forecast log levels `steps` months ahead of the last observation in the
training window.

- **LOCF**: the benchmark.
- **Drift**: last value plus the average monthly log change of the last 12 months.
- **ARIMA(1,1,0)** on the log price.
- **VAR(p)** on log-differences (BIC lag, 1 to 3), using the system's series.
- **VECM, rank 1**, one lagged difference, constant inside the relation.

Systems with a single target series have all five models. The univariate models are
run on every series in a system, on the **same truncated window** as the multivariate
ones, so all five models see identical information.

In [ ]:
HORIZONS = (3, 6, 9, 12)
MIN_TRAIN = 48
VAR_MAX_LAG = 3
MODELS = ("LOCF", "Drift", "ARIMA", "VAR", "VECM_r1")


def forecast_locf(train, steps):
    """Carry the last observed log price forward."""
    return pd.DataFrame(
        np.tile(train.iloc[-1].to_numpy(), (steps, 1)), columns=train.columns
    )


def forecast_drift(train, steps):
    """Last value plus the trailing 12-month average log change."""
    drift = (train.iloc[-1] - train.iloc[-13]) / 12
    path = train.iloc[-1].to_numpy() + np.outer(
        np.arange(1, steps + 1), drift.to_numpy()
    )
    return pd.DataFrame(path, columns=train.columns)


def forecast_arima(train, steps):
    """ARIMA(1,1,0) fitted separately to each column."""
    out = {}
    for col in train.columns:
        out[col] = ARIMA(train[col].to_numpy(), order=(1, 1, 0)).fit().forecast(steps)
    return pd.DataFrame(out)


def forecast_var(train, steps):
    """VAR on log-differences; returns log-level forecasts."""
    diffs = train.diff().dropna()
    lag = max(int(VAR(diffs).select_order(maxlags=VAR_MAX_LAG).bic), 1)
    fit = VAR(diffs).fit(lag)
    delta = fit.forecast(diffs.to_numpy()[-lag:], steps=steps)
    return pd.DataFrame(
        train.iloc[-1].to_numpy() + np.cumsum(delta, axis=0), columns=train.columns
    )


def forecast_vecm_r1(train, steps):
    """Rank-1 VECM on log levels (a VAR in differences needs no rank; this needs n>=2)."""
    fit = VECM(train, k_ar_diff=1, coint_rank=1, deterministic="co").fit()
    return pd.DataFrame(fit.predict(steps=steps), columns=train.columns)


FORECASTERS = {
    "LOCF": forecast_locf,
    "Drift": forecast_drift,
    "ARIMA": forecast_arima,
    "VAR": forecast_var,
    "VECM_r1": forecast_vecm_r1,
}

## 5. Rolling-origin backtest with publication lags

For every decision month, the models are fitted on data up to
`decision month - offset` and forecast `h + offset` steps ahead, so the target month is
the decision month plus `h`. A fit that fails is excluded for **all** models at that
origin, so all models are scored on identical windows.

In [ ]:
def run_backtest():
    """Backtest all systems with publication lags; returns records and failures."""
    records, failures = [], []
    for system, cols in SYSTEMS.items():
        data = logs[cols].dropna()
        offset = SYSTEM_OFFSET[system]
        steps = max(HORIZONS) + offset
        origins = data.index[MIN_TRAIN + offset - 1 : -min(HORIZONS)]
        for origin in origins:
            train = data.loc[: origin - pd.DateOffset(months=offset)]
            try:
                fc = {name: f(train, steps) for name, f in FORECASTERS.items()}
            except Exception as exc:  # numerical failure: skip origin for all models
                failures.append(
                    {"system": system, "origin": origin, "error": repr(exc)}
                )
                continue
            for h in HORIZONS:
                target = origin + pd.DateOffset(months=h)
                if target not in data.index:
                    continue
                for model, frame in fc.items():
                    for series in cols:
                        records.append(
                            {
                                "system": system,
                                "series": series,
                                "origin": origin,
                                "h": h,
                                "model": model,
                                "forecast": frame.loc[h + offset - 1, series],
                                "actual": data.loc[target, series],
                            }
                        )
    return pd.DataFrame(records), failures


results, failures = run_backtest()
results["error"] = results["forecast"] - results["actual"]  # log points
print(f"{len(results):,} forecasts | failed origins: {len(failures)}")
if failures:
    print(pd.DataFrame(failures).groupby("system").size())
results.groupby("system")["origin"].agg(["min", "max", "nunique"])

## 6. Results

**Relative MAE** = model MAE / LOCF MAE on identical windows; below 1.0 beats LOCF.
The p-value tests equal squared-error loss versus LOCF with HAC standard errors
(overlapping horizons), and is rough: effective samples are small and we run many
comparisons.

In [ ]:
def dm_pvalue(loss_diff, h):
    """HAC p-value for zero mean loss differential versus LOCF."""
    x = np.asarray(loss_diff, dtype=float)
    if len(x) < 10 or np.allclose(x, 0):
        return np.nan
    fit = sm.OLS(x, np.ones(len(x))).fit(cov_type="HAC", cov_kwds={"maxlags": h})
    return fit.pvalues[0]


rows = []
for (system, series, h), grp in results.groupby(["system", "series", "h"]):
    wide = grp.pivot(index="origin", columns="model", values="error")
    abs_err = wide.abs()
    locf_mae = abs_err["LOCF"].mean()
    for model in MODELS[1:]:
        rows.append(
            {
                "system": system,
                "series": series,
                "name": NAMES[series],
                "h": h,
                "model": model,
                "n": len(wide),
                "LOCF MAE": locf_mae,
                "MAE": abs_err[model].mean(),
                "rel MAE": abs_err[model].mean() / locf_mae,
                "win rate": (abs_err[model] < abs_err["LOCF"]).mean(),
                "DM p": dm_pvalue(wide[model] ** 2 - wide["LOCF"] ** 2, h),
            }
        )
scores = pd.DataFrame(rows)

# Which are the targets we care about? The licensed price series, not the drivers.
TARGETS = [
    "MEPS_CR_DE",
    "CRU_CR_DE",
    "MEPS_HDG_DE",
    "CRU_HDG_DE",
    "SS304_DE",
    "EPOXY",
    "S09",
]
target_scores = scores[scores["series"].isin(TARGETS)]
print("Typical LOCF error (mean absolute log error, about % error)")
(
    target_scores.drop_duplicates(["system", "series", "h"])
    .pivot_table(index=["system", "series", "name"], columns="h", values="LOCF MAE")
    .round(3)
)

In [ ]:
rel = target_scores.pivot_table(
    index=["system", "series", "name"], columns=["model", "h"], values="rel MAE"
)
print("Relative MAE vs LOCF (below 1.00 = better than LOCF), publication lags")
rel.round(2)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5), sharey=True)
for ax, model in zip(axes, MODELS[1:], strict=True):
    mat = rel[model]
    im = ax.imshow(mat.to_numpy(), cmap="RdBu_r", vmin=0.7, vmax=1.3, aspect="auto")
    ax.set_xticks(range(len(mat.columns)), [f"h={h}" for h in mat.columns])
    ax.set_yticks(range(len(mat)), [f"{sy}: {s}" for sy, s, _ in mat.index])
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, f"{mat.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
    ax.set_title(f"{model}")
fig.colorbar(im, ax=axes, label="relative MAE (blue = better than LOCF)", shrink=0.8)
plt.show()

In [ ]:
(
    target_scores.groupby(["model", "h"])
    .agg(
        median_rel_MAE=("rel MAE", "median"),
        share_beating_LOCF=("rel MAE", lambda x: (x < 1).mean()),
    )
    .round(3)
    .join(
        target_scores.assign(
            sig_better=lambda d: (d["DM p"] < 0.05) & (d["rel MAE"] < 1),
            sig_worse=lambda d: (d["DM p"] < 0.05) & (d["rel MAE"] >= 1),
        )
        .groupby(["model", "h"])[["sig_better", "sig_worse"]]
        .mean()
        .round(3)
    )
)

In [ ]:
print("DM p-values vs LOCF (rough). A small p alone does not mean a gain:")
print("check that relative MAE is also below 1.")
display(
    target_scores.pivot_table(
        index=["system", "series", "model"], columns="h", values="DM p"
    ).round(3)
)

## 7. Does the free steel PPI help predict the real steel price?

In the VAR/VECM notebook, the free `S09` EU steel PPI was the only series that
clearly beat LOCF at short horizons. Here we put it in a two-variable system with the
real MEPS CR coil price. This tells us two things on identical windows: whether the
PPI is easier to forecast than the price it is meant to proxy, and whether adding the
PPI improves the forecast of the real price (compare `MEPS_CR_DE` rows here with the
`steel_flat` rows above).

In [ ]:
link = scores[scores["system"] == "steel_proxy_link"].pivot_table(
    index=["series", "name", "model"], columns="h", values="rel MAE"
)
print("Relative MAE vs LOCF in the PPI + real CR coil system (below 1.00 beats LOCF)")
display(link.round(2))
print("Same DM p-values")
scores[scores["system"] == "steel_proxy_link"].pivot_table(
    index=["series", "model"], columns="h", values="DM p"
).round(3)

In [ ]:
best = (
    target_scores.groupby(["system", "series", "h"])["rel MAE"]
    .min()
    .unstack("h")
    .round(2)
)
best["avg"] = best.mean(axis=1).round(2)
print(
    "Best relative MAE achieved by any model (a favourable view, picked after the fact)"
)
best

## 8. How wrong is LOCF? An empirical error band

Since LOCF is the point forecast that holds up, the useful question is how wide the
miss typically is. For each series and horizon (including the publication-lag gap) we
take the backtest errors `actual / forecast - 1` and report their 10th, 50th and 90th
percentiles. Read it as: "when LOCF said X, the outcome was between X(1+P10) and
X(1+P90) 80% of the time in this sample". This is the same idea as the repo's LOCF
error-quantile method, in miniature. Origins overlap and regimes shift, so the band is
indicative, not a guarantee. Only the main system for each series is used.

In [ ]:
PRIMARY = {"steel_flat", "stainless_nickel", "epoxy_chain"}
locf = results[
    (results["model"] == "LOCF")
    & results["system"].isin(PRIMARY)
    & results["series"].isin(TARGETS)
].copy()
locf["pct"] = np.exp(-locf["error"]) - 1  # actual / forecast - 1
band = (
    locf.groupby(["series", "h"])["pct"]
    .quantile([0.1, 0.5, 0.9])
    .mul(100)
    .unstack()
    .round(1)
    .rename(columns={0.1: "P10 %", 0.5: "P50 %", 0.9: "P90 %"})
)
band["n origins"] = locf.groupby(["series", "h"])["origin"].nunique()
band

## 9. Forecasts for the September 2026 decision

The last data we may use for the September 30, 2026 decision is the decision month
minus the offset. Shown as the % change from the **last available** observation for the
repo's target months. Treat every non-LOCF number with the backtest in mind.

In [ ]:
target_months = {3: "Dec 2026", 6: "Mar 2027", 9: "Jun 2027"}
decision = pd.Timestamp("2026-09-01")
rows = []
for system, cols in SYSTEMS.items():
    offset = SYSTEM_OFFSET[system]
    train = logs[cols].dropna().loc[: decision - pd.DateOffset(months=offset)]
    steps = max(target_months) + offset
    fc = {name: f(train, steps) for name, f in FORECASTERS.items() if name != "LOCF"}
    for h, month in target_months.items():
        for s in cols:
            if s not in TARGETS or s == "S09" or system not in PRIMARY:
                continue
            last = train[s].iloc[-1]
            row = {
                "series": s,
                "target": month,
                "last available": train.index[-1].strftime("%b %Y"),
            }
            for model, frame in fc.items():
                row[f"{model} %"] = 100 * (
                    np.exp(frame.loc[h + offset - 1, s] - last) - 1
                )
            rows.append(row)
pd.DataFrame(rows).round(1)

## 10. Takeaways

Numeric results are deliberately not stored in this notebook, because they are computed
from licensed and confidential price data. Run it to regenerate them.

What to read from the outputs:

1. Compare each model's relative MAE against LOCF, with its Diebold-Mariano p-value,
   for every series and horizon.
2. Check whether the 12-month drift forecasts in Section 9 are credible before using
   them; compare them with the empirical LOCF error band in Section 8.
3. Compare the licensed series with their free proxies (for example the EU steel PPI)
   before using a proxy as a stand-in.
4. Where no model reliably beats LOCF, use LOCF as the point forecast and report its
   empirical error band (Section 8) rather than a point forecast.

**Caveats:** publication lags for the licensed and epoxy series are assumptions (1
month); epoxy is a calculated cost build, not a market quote; the samples are short
and overlapping; and MEPS/CRU are licensed, so check permissions before sharing any
output. Record the run date and data cut-off alongside any figures taken from it.
